In [1]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms.huggingface_endpoint import HuggingFaceEndpoint
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_community.llms import VLLM

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate

from datasets import Dataset
import os

In [2]:
llm = VLLM(
    model="/root/Qwen1.5-0.5B-Chat",
    max_new_tokens=512,
    top_k=10,
    top_p=0.95,
    temperature=0.8,
)

INFO 04-15 21:54:39 llm_engine.py:74] Initializing an LLM engine (v0.4.0.post1) with config: model='/root/Qwen1.5-0.5B-Chat', tokenizer='/root/Qwen1.5-0.5B-Chat', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, seed=0)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 04-15 21:54:41 selector.py:51] Cannot use FlashAttention because the package is not found. Please install it for better performance.
INFO 04-15 21:54:41 selector.py:25] Using XFormers backend.
INFO 04-15 21:54:43 model_runner.py:104] Loading model weights took 0.9018 GB
INFO 04-15 21:54:44 gpu_executor.py:94] # GPU blocks: 12806, # CPU blocks: 2730
INFO 04-15 21:54:49 model_runner.py:791] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 04-15 21:54:49 model_runner.py:795] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 04-15 21:54:56 model_runner.py:867] Graph capturing finished in 7 secs.


In [3]:
embed_model = HuggingFaceEmbeddings(model_name="/root/bge-large-zh-v1.5",
                                    multi_process=True,
                                    encode_kwargs={'normalize_embeddings': True}) # BAAI/bge-large-zh-v1.5

/home/vipuser/miniconda3/lib/python3.8/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
langchain_llm = LangchainLLMWrapper(llm)

langchain_embeddings = LangchainEmbeddingsWrapper(embed_model)

data_samples = {
    'question': ['中国第一部电影是什么时候拍的？'],
    'answer': ['中国第一部电影拍摄于1905年'],
    'contexts': [['中国电影产业始于1905年，电影《定军山》是中国早期的电影作品之一。', '那个时代，...电影技术开始在中国传播。']],
    'ground_truth': ['中国第一部电影拍摄于1905年']
}

dataset = Dataset.from_dict(data_samples)


In [5]:
# 评估指标
from ragas.metrics import (
    context_precision,
)

result = evaluate(
    dataset,
    llm=langchain_llm,
    embeddings=langchain_embeddings,
    metrics=[context_precision]
)


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  4.15it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 12.43it/s]


In [7]:
print(result.to_pandas()[['question', 'ground_truth', 'context_precision']])

          question     ground_truth  context_precision
0  中国第一部电影是什么时候拍的？  中国第一部电影拍摄于1905年                1.0
